#  Autoregressive Text Generation with Hugging Face

**AIML ZG536 · Large Language Models for Generative AI**

Every large language model you will meet this term — GPT, Llama, Qwen, DeepSeek — is, at its core, doing one thing: **predicting the next token given everything before it**, appending that prediction to the sequence, and repeating. That loop is *autoregressive generation*.

**Learning objectives** — by the end of this lab you will be able to:

1. **Set up** one notebook to run on both **Colab** and **Kubeflow** by detecting the environment at runtime.
2. **Tokenize** text into IDs and back, and explain why the model only ever sees integers.
3. **Read** the next-token probability distribution from a single forward pass (`logits` → softmax).
4. **Generate** text two ways — a hand-written greedy loop and `.generate()` — and explain why they agree.
5. **Compare** decoding strategies (greedy, no-repeat, sampling with temperature / top-k / top-p) and tell repetition apart from factual error.
6. **Distinguish** base, instruct, and chat models, and use a **chat template** to make an instruct model follow an instruction and hold a multi-turn conversation.

We use **PyTorch** throughout. Sections 1–8 use a small base model (`gpt2`, 124M) so the raw mechanics are visible and load in seconds; Section 9 scales up to `Qwen2.5-Instruct`. The mechanics are *identical* for a 70B model.


## 1. Where are we running?

The same notebook should run on **Google Colab** and on our **Kubeflow** cluster without edits. The two differ in how packages are installed and where files live, so the first thing we do is detect the environment and branch on it.

In [ ]:
import os, sys

def detect_environment():
    # Colab injects the `google.colab` module into every runtime.
    try:
        import google.colab  # noqa: F401
        return "colab"
    except ImportError:
        pass
    # Kubeflow notebooks set NB_PREFIX; pods also see the k8s service host var.
    if "NB_PREFIX" in os.environ or "KUBERNETES_SERVICE_HOST" in os.environ:
        return "kubeflow"
    return "local"

ENV = detect_environment()
print(f"Detected environment: {ENV}")

Detected environment: colab


### Environment-specific setup

The two environments have different install rules, so we branch here:

- **Colab** — PyTorch is preinstalled; we just add `transformers` (quiet).
- **Kubeflow** — `transformers` is **version-pinned** in the image. We *use what's there* and never upgrade it
- **local** — install only if missing.

In [ ]:
if ENV == "colab":
    # `!pip` is fine here — IPython applies it even inside the `if` block.
    !pip install -q transformers
    print("Colab: ensured transformers is installed.")
elif ENV == "kubeflow":
    print("Kubeflow: using the pre-installed, version-pinned transformers (no upgrade).")
else:
    try:
        import transformers  # noqa: F401
    except ImportError:
        import subprocess
        subprocess.run([sys.executable, "-m", "pip", "install", "transformers", "torch"], check=False)
    print("Local: transformers available.")

Colab: ensured transformers is installed.


In [ ]:
import torch
import transformers

# Use the GPU if one is attached; otherwise CPU. gpt2 runs comfortably on CPU.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"PyTorch      : {torch.__version__}")
print(f"Transformers : {transformers.__version__}")
print(f"Device       : {DEVICE}")

PyTorch      : 2.11.0+cu128
Transformers : 5.15.0
Device       : cuda


## 2. The libraries we rely on

Two libraries and two classes do all the heavy lifting in this demo.

| Component | What it is | Why we need it |
|---|---|---|
| **`torch`** (PyTorch) | The numerical/tensor + autograd engine the model runs on. Every weight, input, and output is a `torch.Tensor`. | Runs the forward pass and moves data on/off the GPU. |
| **`transformers`** (Hugging Face) | Library of pretrained models + tokenizers behind one consistent API. | Downloads LLMs, its weights, and its tokenizer with two function calls. |
| **`AutoTokenizer`** | The right tokenizer for a model, chosen automatically from its name. | Converts text ↔ integer token IDs — the model never sees characters. |
| **`AutoModelForCausalLM`** | Loads a **causal** (left-to-right) language model. | Given token IDs, returns a score for *every possible* next token. This is the object that "generates". |

*Causal* / *autoregressive* / *decoder-only* all describe the same thing here: the model may only look **backwards**, so it predicts token *n+1* from tokens *1…n*.

## 3. Load the tokenizer and model

We deliberately use `gpt2` — a **pure** autoregressive language model with no chat template and no instruction tuning. That makes the raw next-token mechanics visible without any wrapping to distract from them.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "gpt2"   # 124M params. Swap for "Qwen/Qwen2.5-3B-Instruct" on a GPU box.

# Download (or load from cache) the tokenizer matched to MODEL_NAME.
# It holds the vocab + merge rules that map text <-> the exact token IDs this model was trained on.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Download the pretrained causal-LM weights + config, build the model, and move it onto DEVICE (GPU/CPU).
# .to(DEVICE) matters because inputs and model must live on the same device or the forward pass errors.
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)


model.eval()   # inference mode: turns off dropout and other train-only behaviour

print(f"Loaded {MODEL_NAME}: {model.num_parameters()/1e6:.0f}M parameters")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Loaded gpt2: 124M parameters


## 4. Text → tokens → text

The model operates on **token IDs**, not words. Let's watch the round trip.

*(GPT-2's tokenizer prints a leading `Ġ` to mark a space before a token — that's how it encodes word boundaries.)*

In [ ]:
text = "The capital of France is"
ids = tokenizer(text, return_tensors="pt").input_ids   # a [1, seq_len] tensor of IDs

print("Text   :", text)
print("Tokens :", tokenizer.convert_ids_to_tokens(ids[0]))
print("IDs    :", ids[0].tolist())
print("Decoded:", tokenizer.decode(ids[0]))

Text   : The capital of France is
Tokens : ['The', 'Ġcapital', 'Ġof', 'ĠFrance', 'Ġis']
IDs    : [464, 3139, 286, 4881, 318]
Decoded: The capital of France is


## 5. One step of autoregression: predict the next token

Autoregression is just repeatedly predicting the next token. Let's do a **single** step by hand.

We push the token IDs through the model once. The output `logits` holds a raw score for every token in the vocabulary (~50k for GPT-2) at **every** position. We only care about the scores at the **last** position — those predict what comes next. A softmax turns those scores into a probability distribution.

text = "The capital of France is"

In [ ]:
import torch.nn.functional as F

with torch.no_grad():                          # no gradients needed at inference
    logits = model(ids.to(DEVICE)).logits      # shape: [batch, seq_len, vocab_size]

next_token_logits = logits[0, -1, :]           # scores for the token AFTER "is"
probs = F.softmax(next_token_logits, dim=-1)   # -> probabilities over the vocab

topk = torch.topk(probs, 5)
print(f'Prompt: "{text}"\n\nTop 5 candidates for the next token:')
for prob, tid in zip(topk.values, topk.indices):
    print(f"  {tokenizer.decode(tid):<12} {prob.item():.3f}")

Prompt: "The capital of France is"

Top 5 candidates for the next token:
   the         0.085
   now         0.048
   a           0.046
   France      0.032
   Paris       0.032


## 6. The autoregressive loop, by hand

Now the whole idea in one loop: take the most likely next token (**greedy**), **append** it to the sequence, and feed the longer sequence back in. Repeat. This is exactly what `.generate()` does internally.

In [ ]:
generated = ids.clone().to(DEVICE)

for _ in range(20):
    with torch.no_grad():
        logits = model(generated).logits
    next_id = logits[0, -1, :].argmax()              # greedy: the highest-prob token
    next_id = next_id.reshape(1, 1)
    generated = torch.cat([generated, next_id], dim=1)   # append, then loop again
    if next_id.item() == tokenizer.eos_token_id:         # stop if the model ends
        break

print(tokenizer.decode(generated[0], skip_special_tokens=True))

The capital of France is the capital of the French Republic, and the capital of the French Republic is the capital of the French


## 7. The same thing with `.generate()`

In practice you never hand-write that loop — `.generate()` does it for you, plus key-value caching, batching, and stop conditions. Here it runs **greedy** decoding, identical in spirit to the loop above.

In [ ]:
with torch.no_grad():
    out = model.generate(
        ids.to(DEVICE),
        max_new_tokens=20,
        do_sample=False,                     # greedy — deterministic, no randomness
        pad_token_id=tokenizer.eos_token_id, # gpt2 has no pad token; reuse eos
    )
print(tokenizer.decode(out[0], skip_special_tokens=True))

The capital of France is the capital of the French Republic, and the capital of the French Republic is the capital of the French


In [ ]:
ids = ids.to(DEVICE)

for label, kwargs in [
    ("greedy",            dict(do_sample=False)),
    ("greedy + no-repeat",dict(do_sample=False, no_repeat_ngram_size=3)),
    ("sampling",          dict(do_sample=True, temperature=0.8, top_p=0.95)),
]:
    out = model.generate(ids, max_new_tokens=30, pad_token_id=tokenizer.eos_token_id, **kwargs)
    print(f"[{label}]\n{tokenizer.decode(out[0], skip_special_tokens=True)}\n")

[greedy]
The capital of France is the capital of the French Republic, and the capital of the French Republic is the capital of the French Republic.

The French Republic is the capital

[greedy + no-repeat]
The capital of France is the capital of the French Republic, and the capital is the city of Paris. The capital of Europe is the European Union. The European Union is the

[sampling]
The capital of France is Paris. The French have a population of 3.5 million. But they also have a long history of ethnic tensions between the French and the indigenous communities



### Comparing three decoding strategies (same prompt: *"The capital of France is"*)

| Strategy | Output | What happened |
|---|---|---|
| **Greedy** | Picks the single highest-probability token every step. Falls into a **self-reinforcing loop** — once inside the phrase, the most likely continuation regenerates the same phrase. This is *neural text degeneration*. |
| **Greedy + no-repeat** | Still **deterministic**, but forbidden from repeating any 3-gram (`no_repeat_ngram_size=3`). The loop is broken — it even gets *Paris* right — but forced off its high-probability path, it drifts into confident nonsense. |
| **Sampling** | **Draws** from the distribution (`temperature`, `top_p`) instead of always taking the top token. Escapes the loop by a different mechanism, reads more naturally, and produces different text each run — still with invented specifics. |

**Takeaways**

- All three continuations are **grammatical**. The repetition loop is a **decoding artifact**, fixable two different ways (forbid repeats *or* add randomness).
- None of the fixes make a 124M model **factually reliable** — decoding controls *how* the model generates, not *what it knows*. **Fluency ≠ correctness.**
- This is why production systems don't feed a bare prompt to a base model: they use an **instruction-tuned** model with a **chat template**, and **ground** it (RAG) when facts matter

## 8. Making it less robotic: sampling

Greedy always takes the single most likely token, so it is deterministic and often repetitive. **Sampling** instead *draws* from the probability distribution. Three knobs shape that draw:

- **temperature** — flattens the distribution (`>1`, more random) or sharpens it (`<1`, more focused).
- **top-k** — restricts sampling to the `k` most likely tokens.
- **top-p** (nucleus) — restricts sampling to the smallest set of tokens whose probabilities sum to `p`.

Run the cell a few times — with sampling on, each run gives a different continuation. (Uncomment the seed line to make a run reproducible.)

In [ ]:
# torch.manual_seed(0)   # uncomment for a repeatable result

prompt = "Once upon a time"
p_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(DEVICE)

with torch.no_grad():
    out = model.generate(
        p_ids,
        max_new_tokens=40,
        do_sample=True,        # turn sampling on
        temperature=0.8,
        top_k=50,
        top_p=0.95,
        pad_token_id=tokenizer.eos_token_id,
    )
print(tokenizer.decode(out[0], skip_special_tokens=True))

Once upon a time, there was a new era of adventure in the gaming world, a new era of social games, a new era of games that don't have to be tied to a game's world and are more


## Recap — the autoregressive core (base model)

- A causal LM turns a sequence of tokens into a **probability distribution over the next token**.
- **Autoregressive generation** = pick/sample a token → append → repeat.
- `.generate()` is that loop plus optimizations. The **decoding strategy** (greedy vs sampling) and its knobs shape the output — the model weights never change.

**Scaling up:** change `MODEL_NAME = "gpt2"` to an instruction-tuned model such as `Qwen/Qwen2.5-3B-Instruct` on a GPU. Everything above stays the same — you'd simply wrap your prompt in the model's **chat template** first.

## 9. From a base model to instruct & chat models

`gpt2` is a **base** model — trained only to predict the next token on raw web text. It *continues* text; it doesn't *answer*. To get an assistant, the model must be fine-tuned further. Three terms you'll hear:

| Term | Trained on | Behaviour |
|---|---|---|
| **Base / pretrained** (`gpt2`) | Raw text, next-token only | Continues your text |
| **Instruct** (`…-Instruct`) | *(instruction → response)* pairs | Follows a single instruction |
| **Chat** | Multi-turn dialogues with **roles** (system / user / assistant) | Holds a conversation |

In modern practice these have merged: a model like **`Qwen2.5-3B-Instruct`** is trained on chat-formatted data, so it is *both* an instruction-follower and a chat model. The mechanism that makes it work is the **chat template** — the exact special-token format the model was trained on. Feed it a bare prompt and it underperforms; feed it the template and it behaves.

### First, see the problem: a base model can't take an instruction

We reuse the `gpt2` `model`/`tokenizer` already loaded above. Ask it a direct question and watch it *continue* the text instead of *answering* — this is the gap an instruct model fills.

In [ ]:
# Ask the BASE model (gpt2) a direct question. It won't answer — it just continues text,
# because it was never trained to treat the input as an instruction.
base_prompt = "What is the capital of France?"
base_ids = tokenizer(base_prompt, return_tensors="pt").input_ids.to(DEVICE)

out = model.generate(base_ids,
                     max_new_tokens=30,
                     do_sample=False,
                     pad_token_id=tokenizer.eos_token_id)

print(tokenizer.decode(out[0], skip_special_tokens=True))

What is the capital of France?

The capital of France is Paris.

The capital of France is Paris.

The capital of France is Paris.

The


### Load an instruct / chat model

We load `Qwen2.5-3B-Instruct` — the same family we use for the rest of the course. On CPU it would be slow, so we fall back to the 0.5B version there; the API and behaviour are identical, just smaller.

In [ ]:
# Pick a size the hardware can handle: 3B on GPU, 0.5B on CPU. Same API either way.
CHAT_MODEL = "Qwen/Qwen2.5-3B-Instruct" if DEVICE == "cuda" else "Qwen/Qwen2.5-0.5B-Instruct"

chat_tok = AutoTokenizer.from_pretrained(CHAT_MODEL)
chat_model = AutoModelForCausalLM.from_pretrained(CHAT_MODEL).to(DEVICE)

# On GPU, use fp16 to halve memory. .half() is version-proof (unlike the torch_dtype arg);
# we skip it on CPU where fp16 is slow.
if DEVICE == "cuda":
    chat_model = chat_model.half()
chat_model.eval()

print(f"Loaded {CHAT_MODEL}: {chat_model.num_parameters()/1e9:.2f}B parameters")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded Qwen/Qwen2.5-3B-Instruct: 3.09B parameters


### The chat template — see what actually gets fed to the model

A chat model expects its input wrapped in special role markers (for Qwen: `<|im_start|>` … `<|im_end|>`). `apply_chat_template` builds that string from a plain list of `{"role", "content"}` messages. We set `tokenize=False` first so we can **read** the formatted prompt before generating from it.

In [ ]:
# A conversation is just a list of role/content dicts.
messages = [
    {"role": "user",
     "content": "What is the capital of France? Answer in one sentence."}
]

# Build the formatted prompt WITHOUT tokenizing, so we can print and inspect it.
# add_generation_prompt=True appends the "assistant" opener so the model knows it's its turn.
prompt_text = chat_tok.apply_chat_template(messages,
                                           tokenize=False,
                                           add_generation_prompt=True)
print(" Prompt:  \n", prompt_text)
# Note the system message Qwen inserts automatically, and the <|im_start|>/<|im_end|> role markers.

 Promp:  
 <|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
What is the capital of France? Answer in one sentence.<|im_end|>
<|im_start|>assistant



### Single-turn: instruction following

Now the real thing. Same template, but tokenized this time.

- **`apply_chat_template(..., return_tensors="pt")`** does the formatting *and* the tokenizing in one step.
- We **slice off the prompt tokens** before decoding, so we print only the model's *new* reply — not the whole prompt echoed back.

In [ ]:
messages = [
    {"role": "user",
     "content": "What is the capital of France? Answer in one sentence."}
]

# Format + tokenize in one call. inputs is a [1, prompt_len] tensor on DEVICE.
inputs = chat_tok.apply_chat_template( messages,
                                      add_generation_prompt=True,
                                      return_tensors="pt"
                                     ).to(DEVICE)

with torch.no_grad():
    # Pass inputs["input_ids"] and inputs["attention_mask"] to generate
    out = chat_model.generate(inputs["input_ids"],
                              attention_mask=inputs["attention_mask"],
                              max_new_tokens=64,
                              do_sample=False)  # greedy


# Full sequence: the formatted prompt (with role markers) plus the model's reply.
print("Full sequence:")
print(chat_tok.decode(out[0], skip_special_tokens=True))


# out[0] holds prompt + reply. Keep only tokens AFTER the prompt -> the reply alone.
# Use inputs["input_ids"].shape[-1] for the length of the prompt tokens.
reply = chat_tok.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
print("\n\n ----------------Now only reply as we do not want the whole prompt----------")
print(reply)


system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
What is the capital of France? Answer in one sentence.
assistant
The capital of France is Paris.


 ----------------Now only reply as we do not want the whole prompt----------
The capital of France is Paris.


### Multi-turn: an actual conversation

The "chat" ability is memory across turns. We add a **system** message to set behaviour, then append each reply back into the `messages` list and ask a follow-up. The follow-up ("what river runs through it?") only resolves if the model still remembers we were talking about Paris.

First we wrap the single-turn logic into a small helper so we don't repeat ourselves.

In [ ]:
def chat(messages, max_new_tokens=96):
    """Format the whole conversation, generate, and return ONLY the new assistant reply."""
    inputs = chat_tok.apply_chat_template( messages,
                                          add_generation_prompt=True,
                                           return_tensors="pt"
    ).to(DEVICE)

    with torch.no_grad():
        out = chat_model.generate(inputs["input_ids"],
                                  attention_mask=inputs["attention_mask"],
                                  max_new_tokens=max_new_tokens,
                                  do_sample=False)
    return chat_tok.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)


# System message sets the persona/rules; user message is the first question.
messages = [
    {"role": "system", "content": "You are a concise geography tutor. Answer in one short sentence."},
    {"role": "user",   "content": "What is the capital of France?"},
]
reply = chat(messages)
print("Assistant:", reply)

# Append the reply, then ask a follow-up that depends on the previous turn.
messages.append({"role": "assistant", "content": reply})
messages.append({"role": "user", "content": "And what river runs through it?"})

reply = chat(messages)
print("Assistant:", reply)   # resolves "it" -> Paris only because the history is in the prompt

Assistant: The capital of France is Paris.
Assistant: The River Seine runs through Paris.


### The shortcut: `pipeline`

Everything above — apply template, tokenize, generate, slice, decode — is bundled into the high-level `pipeline`. Pass it a `messages` list and it does the rest. We build it from the model we already loaded, so there's no second download.

In [ ]:
from transformers import pipeline

# Reuse the loaded model + tokenizer (no re-download, no extra memory copy).
chatbot = pipeline("text-generation", model=chat_model, tokenizer=chat_tok)

messages = [{"role": "user", "content": "Explain autoregressive text generation in one sentence."}]
out = chatbot(messages, max_new_tokens=64, do_sample=False)

# The pipeline returns the full conversation; the last message is the assistant's reply.
print(out[0]["generated_text"][-1]["content"])

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Autoregressive text generation is a method where each word in a sequence is predicted based on the entire prefix of words that have been generated so far, using a model that learns to predict the next word given all previous words.


## Recap — base → instruct → chat

- A **base** model (`gpt2`) only continues text — it can't take an instruction.
- An **instruct / chat** model (`Qwen2.5-Instruct`) is a base model fine-tuned to follow instructions and hold conversations — but only when you feed it its **chat template**.
- The template turns a list of `{role, content}` messages into the exact special-token format the model was trained on. `apply_chat_template` builds it; `pipeline` hides it.
- Decoding strategy  still applies on top — greedy for reproducible answers, sampling for variety.

## Practice questions

Work through these after you've run every cell above. They come in two groups: **understanding checks** (short written answers), **code tasks** (change a cell and observe).

### A. Check your understanding

1. **Tokens, not words.** In Section 4, `"The capital of France is"` became a list of integer IDs. Why does the model never receive the characters `F-r-a-n-c-e`, and what is the `Ġ` symbol doing in the printed token list?
2. **Which position predicts the next token?** The forward pass returns `logits` of shape `[1, seq_len, vocab_size]`. Explain why we read `logits[0, -1, :]` and not `logits[0, 0, :]` to get the *next* token.
3. **Why greedy loops.** In one sentence, explain why greedy decoding got stuck repeating "the capital of the French Republic," while sampling did not.
4. **Decoding fix vs. knowledge limit.** `no_repeat_ngram_size=3` stopped the repetition but produced "the capital of Europe is the European Union." What does this reveal about the difference between fixing *how* a model generates and fixing *what it knows*?
5. **Continue vs. answer.** In Section 9, `gpt2` *continued* "What is the capital of France?" instead of answering it. Why? What kind of additional training turns a base model into one that answers?
6. **What the template buys you.** Looking at the printed output in Section 9, what does `apply_chat_template` actually produce, and why does passing a bare string to a chat model underperform?
7. **Conversation memory.** In the multi-turn cell, why must the assistant's previous reply be appended back into `messages` before the follow-up "what river runs through it?" can resolve to *Paris*?

### B. Code tasks

1. **Temperature sweep.** Re-run the Section 8 sampling cell with `temperature=0.2`, then `temperature=1.5`. Describe how the output changes, and connect it to "sharpening vs. flattening the distribution."
2. **Determinism.** Uncomment `torch.manual_seed(0)` and run the sampling cell twice — confirm the output is identical. Remove the seed and run twice more. What does this demonstrate about `do_sample=True`?
3. **Watch it generate.** Modify the hand-written greedy loop (Section 6) to `print` each new token *as it is produced*, so you can watch generation happen one token at a time.
4. **Roles steer behaviour.** In the multi-turn cell, change the system message to `"You are a pirate. Answer in pirate speak."` and re-run. The model weights didn't change — so what did?